# 01 - ROI Crop Extraction From Ground-Truth Boxes

Extrae crops de objetos usando exclusivamente las anotaciones ground-truth en formato YOLO. No se usan detecciones predichas.

In [1]:
%pip install kagglehub pandas Pillow tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import pandas as pd
import kagglehub
from PIL import Image
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CROPS_DIR = PROJECT_ROOT / "data" / "crops"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CROPS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]
PADDING_RATIO = 0.05

CLASS_PROMPT_NAMES = {
    "0": "gun",
    "1": "knife",
    "2": "pliers",
    "3": "scissors",
    "4": "wrench",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dataset directory: {RAW_DIR}")

Project root: C:\Users\Juan Ramirez\Documents\CLIPmod\clip_module
Raw dataset directory: C:\Users\Juan Ramirez\Documents\CLIPmod\clip_module\data\raw


c:\Users\Juan Ramirez\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def resolve_dataset_root():
    local_candidate = RAW_DIR
    if all((local_candidate / split / "images").exists() and (local_candidate / split / "labels").exists() for split in SPLITS):
        return local_candidate

    kaggle_path = Path(kagglehub.dataset_download("orvile/x-ray-baggage-anomaly-detection"))
    candidates = [kaggle_path] + [path for path in kaggle_path.rglob("*") if path.is_dir()]
    for candidate in candidates:
        if all((candidate / split / "images").exists() and (candidate / split / "labels").exists() for split in SPLITS):
            return candidate
    raise FileNotFoundError(f"Could not find train/valid/test images and labels under {kaggle_path}")


DATASET_ROOT = resolve_dataset_root()
print(f"Using dataset root: {DATASET_ROOT}")


def path_for_metadata(path: Path):
    path = path.resolve()
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def find_image_for_label(images_dir: Path, label_path: Path):
    for ext in IMAGE_EXTENSIONS:
        candidate = images_dir / f"{label_path.stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def yolo_to_xyxy(x_center, y_center, width, height, image_width, image_height, padding_ratio=0.0):
    x1 = (x_center - width / 2) * image_width
    y1 = (y_center - height / 2) * image_height
    x2 = (x_center + width / 2) * image_width
    y2 = (y_center + height / 2) * image_height

    pad_x = (x2 - x1) * padding_ratio
    pad_y = (y2 - y1) * padding_ratio
    x1 = max(0, int(round(x1 - pad_x)))
    y1 = max(0, int(round(y1 - pad_y)))
    x2 = min(image_width, int(round(x2 + pad_x)))
    y2 = min(image_height, int(round(y2 + pad_y)))
    return x1, y1, x2, y2


def read_yolo_label(label_path: Path):
    rows = []
    with label_path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                raise ValueError(f"Invalid YOLO row in {label_path}, line {line_number}: {line!r}")
            class_id = int(float(parts[0]))
            x_center, y_center, width, height = map(float, parts[1:])
            rows.append((class_id, x_center, y_center, width, height))
    return rows

Using dataset root: C:\Users\Juan Ramirez\.cache\kagglehub\datasets\orvile\x-ray-baggage-anomaly-detection\versions\1


In [4]:
required_columns = [
    "crop_id", "split", "image_path", "label_path", "crop_path", "class_id", "class_name",
    "x1", "y1", "x2", "y2", "bbox_width", "bbox_height", "bbox_area",
    "image_width", "image_height", "relative_area",
]

records = []

for split in SPLITS:
    images_dir = DATASET_ROOT / split / "images"
    labels_dir = DATASET_ROOT / split / "labels"
    split_crop_dir = CROPS_DIR / split
    split_crop_dir.mkdir(parents=True, exist_ok=True)

    if not labels_dir.exists():
        print(f"Skipping split '{split}': missing {labels_dir}")
        continue

    label_paths = sorted(labels_dir.glob("*.txt"))
    print(f"{split}: {len(label_paths)} label files")

    for label_path in tqdm(label_paths, desc=f"Extracting {split} crops"):
        image_path = find_image_for_label(images_dir, label_path)
        if image_path is None:
            print(f"Warning: no image found for {label_path}")
            continue

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image_width, image_height = image.size
            annotations = read_yolo_label(label_path)

            for object_index, (class_id, x_center, y_center, width, height) in enumerate(annotations):
                class_key = str(class_id)
                if class_key not in CLASS_PROMPT_NAMES:
                    raise KeyError(f"Unknown class_id={class_id} in {label_path}")
                class_name = CLASS_PROMPT_NAMES[class_key]

                x1, y1, x2, y2 = yolo_to_xyxy(
                    x_center, y_center, width, height, image_width, image_height, PADDING_RATIO
                )
                bbox_width = max(0, x2 - x1)
                bbox_height = max(0, y2 - y1)
                if bbox_width == 0 or bbox_height == 0:
                    print(f"Warning: empty crop skipped for {label_path}, object {object_index}")
                    continue

                crop_id = f"{split}_{label_path.stem}_{object_index:03d}_{class_name}"
                crop_path = split_crop_dir / f"{crop_id}.png"
                crop = image.crop((x1, y1, x2, y2))
                crop.save(crop_path)

                bbox_area = bbox_width * bbox_height
                records.append({
                    "crop_id": crop_id,
                    "split": split,
                    "image_path": path_for_metadata(image_path),
                    "label_path": path_for_metadata(label_path),
                    "crop_path": path_for_metadata(crop_path),
                    "class_id": class_id,
                    "class_name": class_name,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "bbox_width": bbox_width,
                    "bbox_height": bbox_height,
                    "bbox_area": bbox_area,
                    "image_width": image_width,
                    "image_height": image_height,
                    "relative_area": bbox_area / float(image_width * image_height),
                })

metadata = pd.DataFrame(records, columns=required_columns)
metadata_path = OUTPUTS_DIR / "crops_metadata.csv"
metadata.to_csv(metadata_path, index=False)
print(f"Saved {len(metadata)} crops to {CROPS_DIR}")
print(f"Saved metadata to {metadata_path}")
metadata.head()

train: 6181 label files


Extracting train crops: 100%|██████████| 6181/6181 [00:45<00:00, 134.81it/s]


valid: 1766 label files


Extracting valid crops: 100%|██████████| 1766/1766 [00:12<00:00, 141.00it/s]


test: 883 label files


Extracting test crops: 100%|██████████| 883/883 [00:06<00:00, 137.15it/s]


Saved 8830 crops to C:\Users\Juan Ramirez\Documents\CLIPmod\clip_module\data\crops
Saved metadata to C:\Users\Juan Ramirez\Documents\CLIPmod\clip_module\outputs\crops_metadata.csv


,crop_id,split,image_path,label_path,crop_path,class_id,class_name,x1,y1,x2,y2,bbox_width,bbox_height,bbox_area,image_width,image_height,relative_area
0,train_009000_jpg.rf.8c46e1aa5b46a0ad24ee4bcb29...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009000_jpg.rf.8c46e1aa5...,2,pliers,258,210,286,241,28,31,868,416,416,0.005016
1,train_009002_jpg.rf.18bf80f2cfdb51f853da15019f...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009002_jpg.rf.18bf80f2c...,2,pliers,219,125,245,157,26,32,832,416,416,0.004808
2,train_009003_jpg.rf.46963402c4cb6f46a47e508b89...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009003_jpg.rf.46963402c...,2,pliers,153,145,199,171,46,26,1196,416,416,0.006911
3,train_009007_jpg.rf.a5143afbb0c741f3b60fc72403...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009007_jpg.rf.a5143afbb...,2,pliers,154,105,180,187,26,82,2132,416,416,0.012320
4,train_009012_jpg.rf.bc99877ade8754d2be89119361...,train,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,C:\Users\Juan Ramirez\.cache\kagglehub\dataset...,data\crops\train\train_009012_jpg.rf.bc99877ad...,2,pliers,322,144,347,178,25,34,850,416,416,0.004912


In [5]:
assert list(metadata.columns) == required_columns
assert metadata["crop_id"].is_unique
assert metadata["class_name"].equals(metadata["class_id"].astype(str).map(CLASS_PROMPT_NAMES))
assert (metadata["bbox_width"] > 0).all() and (metadata["bbox_height"] > 0).all()
metadata.groupby(["split", "class_name"]).size().unstack(fill_value=0)

class_name,gun,knife,pliers,scissors,wrench
split,,,,,
test,166,193,118,203,203
train,1284,1393,698,1379,1427
valid,391,389,225,366,395
